# Junction relocation, on by default since 0.5.0

Since dw3d 0.5.0 every reconstruction moves the finished mesh's trijunction vertices onto the
trijunction curves the segmentation mask itself gives. Triangles and material labels are not
touched; only vertex positions on the trijunction lines move.

This notebook reconstructs one timepoint twice — with the default, and with
`relocate_junctions = False` (the 0.4 behaviour) — checks that the two meshes share their
connectivity, and draws one trijunction line from each beside the mask's own curve.

**Input.** Set the environment variable `DW3D_EXAMPLE_MASK` to a label image (`.tif`, exterior `0`),
for example one timepoint of a developmental time series. Without it the notebook falls back to
`../data/Images/3.tif`, which is present in a development checkout only.

In [ ]:
import os
from pathlib import Path

import numpy as np
import skimage.io as io

from dw3d import extract_junction_curves, get_default_mesh_reconstruction_algorithm
from dw3d.junction_curves import mesh_trijunction_components

MASK = Path(os.environ.get("DW3D_EXAMPLE_MASK", "../data/Images/3.tif"))
MIN_DISTANCE = 3  # the default spacing; at 5 or more, CoarseSpacingRelocationWarning is emitted once
mask = io.imread(MASK)
print(MASK.name, mask.shape, len(np.unique(mask)) - 1, "labels")

## 1) Reconstruct with the default, and with relocation off

In [ ]:
default = get_default_mesh_reconstruction_algorithm(min_distance=MIN_DISTANCE)
points, triangles, labels = default.construct_mesh_from_segmentation_mask(mask)

untreated = get_default_mesh_reconstruction_algorithm(min_distance=MIN_DISTANCE)
untreated.relocate_junctions = False  # the 0.4 mesh, exactly
base_points, base_triangles, base_labels = untreated.construct_mesh_from_segmentation_mask(mask)

# Relocation moves vertices only: the two meshes have the same triangles and labels.
assert np.array_equal(triangles, base_triangles) and np.array_equal(labels, base_labels)
provenance = default.relocation_provenance
print("relocation ran:", provenance["ran"], "| warning emitted:", provenance["coarse_spacing_warning_emitted"])
report = provenance["report"]
print(f"{report['n_accepted']} of {report['n_targets']} junction vertices moved; "
      f"self-intersections {report['n_self_intersections_before']} -> {report['n_self_intersections_after']}")

## 2) Pick one trijunction line

Among the lines whose mask curve has exactly one counterpart in the mesh, take the one the
relocation moved most (largest median vertex displacement). It is chosen to make the change
visible, so it is not a typical line.

In [ ]:
curves = extract_junction_curves(mask, base_points, base_triangles, base_labels)
components = mesh_trijunction_components(base_points, base_triangles, base_labels)

best = None
for curve in curves.matched:
    parts = components.get(tuple(int(x) for x in curve.triple), [])
    if len(parts) != 1 or len(parts[0]["vertices"]) < 6:
        continue
    vertices = parts[0]["vertices"]
    moved_by = np.linalg.norm(points[vertices] - base_points[vertices], axis=1)
    if best is None or np.median(moved_by) > best[0]:
        best = (float(np.median(moved_by)), curve, parts[0])
median_move, curve, line = best
print(f"trijunction {tuple(int(x) for x in curve.triple)}: {len(line['vertices'])} vertices, "
      f"median displacement {median_move:.2f} voxels")

## 3) Draw it: relocation off (left) and the 0.5.0 default (right)

In [ ]:
import matplotlib.pyplot as plt  # matplotlib is an optional dependency: pip install matplotlib

def shared_limits(point_sets, margin=1.0):
    """One (low, high) per axis covering every point set, padded to a cube so both panels match."""
    everything = np.vstack(point_sets)
    low, high = everything.min(axis=0) - margin, everything.max(axis=0) + margin
    half = (high - low).max() / 2
    centre = (high + low) / 2
    return [(c - half, c + half) for c in centre]


# Both panels share one set of limits and ticks, so the two lines can be compared by eye.
vertices = line["vertices"]
limits = shared_limits([base_points[vertices], points[vertices], curve.points])
fig = plt.figure(figsize=(10, 4.5))
for panel, (mesh_points, title) in enumerate(((base_points, "relocate_junctions=False (0.4)"),
                                              (points, "default (0.5.0)"))):
    ax = fig.add_subplot(1, 2, panel + 1, projection="3d")
    ax.set_xlim(*limits[0]); ax.set_ylim(*limits[1]); ax.set_zlim(*limits[2])
    for axis, (low, high) in zip((ax.xaxis, ax.yaxis, ax.zaxis), limits):
        axis.set_ticks(np.linspace(low, high, 5).round(1))
    for v1, v2 in line["edges"]:  # the mesh's own trijunction edges
        ax.plot(*np.stack([mesh_points[v1], mesh_points[v2]]).T, color="k", linewidth=2)
    ax.plot(*curve.points.T, color="tab:red", linestyle="--", label="mask trijunction curve")
    ax.set_title(title)
    ax.set_xlabel("x (voxels)"); ax.set_ylabel("y (voxels)"); ax.set_zlabel("z (voxels)")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig("default_junction_relocation.png", dpi=120)
print("saved default_junction_relocation.png")